In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"
import keras
import keras.ops as K
from keras.layers import Input, Flatten, Dense

# from keras.models import Sequential
from deel.lip.model import Sequential

from deel.lip.layers import (
    SpectralDense,
    SpectralConv2D,
    ScaledL2NormPooling2D,
    FrobeniusDense,
)
from deel.lip.activations import GroupSort, GroupSort2
from deel.lip.losses import HKR, KR, HingeMargin, MulticlassHKR, MulticlassKR

import numpy as np
import decomon

import sys

# setting path
sys.path.append('..')

from data_processing import load_data, select_data_for_radius_evaluation_MNIST08
from radius_evaluation_tools import compute_binary_certificate, starting_point_dichotomy
from lipschitz_decomon_tools import echantillonner_boule_l2_simple
from LearnedHyperplaneTools import *

In [2]:
x_train, x_test, y_train, y_test, y_test_ord = load_data("MNIST08")

In [3]:
model_path = "/home/aws_install/robustess_project/lip_models/demo3_FC_vanilla_MNIST08_channelfirst_False_disj_Neurons_single_output.keras"
model_bin = keras.models.load_model(model_path)
model_bin.compile(
   
    loss=HKR(
        alpha=10.0, min_margin=1.0
    ),  # HKR stands for the hinge regularized KR loss
    metrics=[
        # KR,  # shows the KR term of the loss
        HingeMargin(min_margin=1.0),  # shows the hinge term of the loss
    ],
    optimizer=keras.optimizers.Adam(learning_rate=0.001),)

model_bis = keras.models.load_model("/home/aws_install/robustess_project/lip_models/demo3_FC_vanilla_MNIST08_channelfirst_False_disj_Neurons_single_output_converted_4logits.keras")
model_bis.compile(
        # decreasing alpha and increasing min_margin improve robustness (at the cost of accuracy)
        # note also in the case of lipschitz networks, more robustness require more parameters.
        loss=MulticlassHKR(alpha=100, min_margin=0.25),
        optimizer=keras.optimizers.Adam(1e-4),
        metrics=["accuracy", MulticlassKR()],)

images, labels, idx_list = select_data_for_radius_evaluation_MNIST08(x_test, y_test_ord, model_bis)

/home/aws_install/miniconda3/envs/k3torchenv/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 12 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
/home/aws_install/miniconda3/envs/k3torchenv/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 14 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [4]:
pt_choosen = 0
eps = 1.7

In [5]:
x = images[pt_choosen:pt_choosen+1].flatten().detach().cpu().numpy()

In [6]:
y_list = [x]
for i in range(100):
    y_list.append(echantillonner_boule_l2_simple(x, eps))

In [7]:
# y_list = [x]
# for i in range(20):
#     y_list.append(echantillonner_boule_l2_simple(x, eps, True))

In [ ]:
_, f = get_local_maximum_Learned_Hyperplane(x, labels[pt_choosen], eps, y_list, model_bin, False, 1, 3000, 40, 0.01, 50)

In [10]:
f

0.91532886

In [ ]:
model_bin(y_list[1].reshape((1,28,28))[None])

tensor([[-1.6616]], device='cuda:0', grad_fn=<MmBackward0>)

In [ ]:
nb_pts = 50
pt_choosen = 1
x = images[pt_choosen:pt_choosen+1].flatten().detach().cpu().numpy()
label = labels[pt_choosen]
result = []
# eps = np.abs(model(images[pt_choosen:pt_choosen+1]).detach().cpu().numpy())*1.01
# eps_list = np.linspace(1.66, 1.81, 10)
eps_list = np.linspace(3.03, 3.25, 10)
# eps_list = np.linspace(3.05, 3.35, 10)
# eps_list = np.linspace(4.69, 4.74, 10)
for eps in eps_list:
    print(eps)
    y_list = []
    for i in range(nb_pts):
        ech = echantillonner_boule_l2_simple(x, eps, uniform=False)
        # print(np.linalg.norm(x-ech))
        y_list.append(ech)

    result.append(function_to_optimize_all(echantillonner_boule_l2_simple(x, eps, uniform=False), label, x, y_list, eps, optimization=False, model=model_bin, n_data=5000, n_epochs=20, L=1))

3.03
3.054444444444444
3.078888888888889
3.103333333333333
3.1277777777777778
3.152222222222222
3.1766666666666667
3.201111111111111
3.2255555555555557
3.25


In [ ]:
result

[1.3114188,
 1.3235235,
 1.1675446,
 1.2217767,
 1.1399732,
 1.1674163,
 1.4766192,
 1.32533,
 1.3108587,
 1.4470537]

In [ ]:
result

[0.6781297,
 0.63967407,
 0.60023737,
 0.71914065,
 0.72582865,
 0.7842766,
 0.7816254,
 0.695223,
 0.88805306,
 0.8274647]